# FreePhish implementation

In [ ]:
%cd /content
!rm -f freephish_*.py requirements.txt
from google.colab import files
files.upload()   # select: freephish_old.py, freephish_enhanced.py, freephish_gan.py, requirements.txt
!ls

/content


Saving freephish_gan.py to freephish_gan.py
Saving freephish_enhanced.py to freephish_enhanced.py
Saving freephish_old.py to freephish_old.py
Saving requirements.txt to requirements.txt
adv_results.csv        freephish_gan.py       results_baseline.predictions.csv
adv_results.json       freephish_old.py       results_enhanced.json
detections.jsonl       model_enh.joblib       results_enhanced.predictions.csv
feat_adv_train.npz     model.joblib	      sample_data
feat_enh.npz	       observations.jsonl     sample.parquet
feat_gan_test.npz      __pycache__	      snapshots
feat.npz	       requirements.txt       urls.txt
freephish_enhanced.py  results_baseline.json


In [ ]:
!pip install -q -r requirements.txt

dataset


In [ ]:
!python freephish_old.py download --repo phreshphish/phreshphish --max-shards 20 --per-class 3118 --out sample.parquet

77 parquet files in repo; first few: ['data/test-000.parquet', 'data/test-001.parquet', 'data/test-002.parquet', 'data/test-003.parquet', 'data/test-004.parquet']
shard: data/test-000.parquet | columns: ['sha256', 'url', 'label', 'target', 'date', 'lang', 'lang_score', 'html'] | rows: 1000
first row group label values: ['benign', 'phish']
collected so far: {0: 547, 1: 453}
shard: data/test-001.parquet | columns: ['sha256', 'url', 'label', 'target', 'date', 'lang', 'lang_score', 'html'] | rows: 5236
collected so far: {0: 3118, 1: 2802}
shard: data/test-002.parquet | columns: ['sha256', 'url', 'label', 'target', 'date', 'lang', 'lang_score', 'html'] | rows: 5759
collected so far: {0: 3118, 1: 3118}
collected 6236 rows (~4403 MB). Writing sample.parquet in chunks (progress shown) - DO NOT interrupt until 'saved' appears...
  ...written 311/6236 rows (1s elapsed)
  ...written 622/6236 rows (3s elapsed)
  ...written 933/6236 rows (4s elapsed)
  ...written 1244/6236 rows (5s elapsed)
  ...wr

In [ ]:
import pyarrow.parquet as pq, pandas as pd
print(pq.ParquetFile("sample.parquet").metadata.num_rows, "rows")
print(pd.read_parquet("sample.parquet", columns=["url", "label"])["label"].value_counts())

6236 rows
label
benign    3118
phish     3118
Name: count, dtype: int64


## Stage 1: baseline (original 22 features)


In [ ]:
!python freephish_old.py train --csv sample.parquet --balance --dedup --cache feat.npz

columns: ['url', 'label', 'html'] | rows: 6236
label values found: ['benign', 'phish']
positive (phishing) label(s): {'phish'}
dedup: 6236 -> 5545 rows
class counts: {0: 2427, 1: 2427}
using HTML column: html
loaded cached features: feat.npz
unique domain groups: 3068

== Main run (group split) | test rows sharing a domain with train: 0.0% ==
---- WITHOUT enhanced features (original 22 FreePhish-style features) ----
train time 12.9s
  StackModel n_test=1392       acc 0.935  prec 0.939  rec 0.923  F1 0.931 | total 117.51s  median 0.0545s/URL (n=1392)
saved results -> results_baseline.json and results_baseline.predictions.csv
saved model.joblib


## Stage 2: enhanced
Saves `results_enhanced.json` + `results_enhanced.predictions.csv` + `model_enh.joblib`.

In [ ]:
!python freephish_enhanced.py train --csv sample.parquet --balance --dedup --cache feat_enh.npz

excluded features: ['resp_time']
columns: ['url', 'label', 'html'] | rows: 6236
feature groups: ['freephish', 'url', 'html', 'fwb', 'brand', 'behavior', 'visual']
label values found: ['benign', 'phish']
positive (phishing) label(s): {'phish'}
dedup: 6236 -> 5545 rows
class counts: {0: 2427, 1: 2427}
loaded cached features: feat_enh.npz

== group split | train 3462 / test 1392 | test rows sharing a domain with train: 0.0% ==
train time 37.4s
  StackModel n_test=1392       acc 0.944  prec 0.958  rec 0.923  F1 0.940 | total 203.96s  median 0.1002s/URL (n=1392)
(timing = feature extraction + single-sample inference; preprocessing and page fetch not included)
saved results -> results_enhanced.json and results_enhanced.predictions.csv
saved model_enh.joblib


## Stage 3: compare
Reads the two results files above and prints the side-by-side table (no ROC AUC row, no top-feature dump — trimmed per your earlier requests).

In [ ]:
!python freephish_old.py compare

BASELINE (without enhanced): original 22 (paper-style baseline) | split=group seed=42 | train 3462 / test 1392 | data=sample.parquet | created 2026-11-08T06:47:38
ENHANCED: enhanced | split=group seed=42 | train 3462 / test 1392 | data=sample.parquet | created 2026-11-08T06:48:46

== SIDE-BY-SIDE ==
metric                        WITHOUT enhanced   WITH enhanced   difference
Accuracy                                0.9346          0.9440      +0.0093
Precision                               0.9386          0.9577      +0.0191
Recall                                  0.9230          0.9230      +0.0000
F1-score                                0.9307          0.9400      +0.0093
Total time (s)                          117.51          203.96       +86.45
Median runtime (s/URL)                  0.0545          0.1002      +0.0458


In [ ]:
import os, io, requests, pandas as pd
from urllib.parse import urlparse

FWB_SUFFIXES = [
    "weebly.com", "000webhostapp.com", "blogspot.com", "wixsite.com", "wix.com",
    "sites.google.com", "github.io", "firebaseapp.com", "web.app", "square.site",
    "wordpress.com", "forms.gle", "docs.google.com", "sharepoint.com", "yolasite.com",
    "godaddysites.com", "mailchimpsites.com", "glitch.me", "hpage.com", "zohopublic.com",
]
def is_fwb(url):
    host = (urlparse(url).hostname or "").lower()
    return any(host == s or host.endswith("." + s) for s in FWB_SUFFIXES)

N = 10
UA = {"User-Agent": "phishtank/research-demo"}

# ---- phishing: PhishTank verified+online feed (optional app key in env PHISHTANK_KEY) ----
key = os.environ.get("PHISHTANK_KEY")
pt_url = (f"http://data.phishtank.com/data/{key}/online-valid.csv" if key
          else "http://data.phishtank.com/data/online-valid.csv")
try:
    r = requests.get(pt_url, headers=UA, timeout=60)
    r.raise_for_status()
    phish_urls = pd.read_csv(io.StringIO(r.text))["url"].dropna().astype(str).tolist()
    src = "PhishTank"
except Exception as e:
    print("PhishTank failed:", type(e).__name__, e, "-> falling back to Phishing.Database (GitHub)")
    r = requests.get("https://raw.githubusercontent.com/mitchellkrogza/Phishing.Database/master/phishing-links-ACTIVE.txt", timeout=60)
    phish_urls = [u.strip() for u in r.text.splitlines() if u.strip()]
    src = "Phishing.Database"
fwb_phish = [u for u in phish_urls if is_fwb(u)]
print(f"{src}: {len(phish_urls)} URLs, {len(fwb_phish)} on FWB domains")

# ---- benign: popular public GitHub repos that have a GitHub Pages site ----
benign_urls = []
for page in (1, 2):
    j = requests.get("https://api.github.com/search/repositories",
                      params={"q": "stars:>3000", "sort": "stars", "order": "desc", "per_page": 100, "page": page},
                      headers={"Accept": "application/vnd.github+json"}, timeout=20).json()
    for repo in j.get("items", []):
        if not repo.get("has_pages"):
            continue
        u = f"https://{repo['owner']['login'].lower()}.github.io/{repo['name']}/"
        if u in benign_urls or not is_fwb(u):
            continue
        try:
            rr = requests.get(u, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
            if rr.status_code < 400 and is_fwb(rr.url):
                benign_urls.append(u)
        except Exception:
            pass
        if len(benign_urls) >= N:
            break
    if len(benign_urls) >= N:
        break
print(f"benign FWB pages found: {len(benign_urls)}")

n = min(N, len(fwb_phish), len(benign_urls))
urls_to_watch = fwb_phish[:n] + benign_urls[:n]
with open("urls.txt", "w") as f:
    f.write("\n".join(urls_to_watch))
if os.path.exists("detections.jsonl"):
    os.remove("detections.jsonl")
print(f"wrote {len(urls_to_watch)} URLs to urls.txt")

PhishTank: 48155 URLs, 11479 on FWB domains
benign FWB pages found: 10
wrote 20 URLs to urls.txt


In [ ]:
!python freephish_old.py monitor --model model.joblib --source urls.txt --out detections.jsonl --once

PHISH 0.747 https://munivirtualenlinea.godaddysites.com/
PHISH 0.830 https://munivirtualenlinea.godaddysites.com
ok    0.188 https://munivirtualenlinea.godaddysites.com/firma-digital-municipal
PHISH 0.995 https://lesusdelgado4.wixsite.com/mike-smith-web-servi?fbclid=IwY2xjawVbDiNleHRuA2FlbQIxMQBwZG9mAWJyaWQRMWJ3MWh1aXFYM2xkcTQwWGNzcnRjBmFwcF9pZBAyMjIwMzkxNzg4MjAwODkyAAEfx7-BMHpejvVc8MjMZgb59PFrYOXTblMCQhA5gXFVlaOxeDe0Z8erbs8rm2A_aem__M8-29bZftUv_RmA52rtYA
PHISH 0.953 https://en-trezor-suite-io.square.site/
PHISH 0.995 https://telstramail-kyc-trial.weebly.com/
PHISH 0.995 https://dfjhdfjhdf.weebly.com/
PHISH 0.837 https://sdrsyusiiw11.weebly.com/
PHISH 0.523 https://felixdelgado220618.wixsite.com/comercio-digital?fbclid=IwY2xjawVYaMdleHRuA2FlbQIxMQBwZG9mAWJyaWQRMUh3N2h3SzFTY2h3RkVyZ0dzcnRjBmFwcF9pZBAyMjIwMzkxNzg4MjAwODkyAAEfq3XQ9B4QfzlXBCAEPFqkOws_mEgpWTWfCnq2tHVYoYuliyc5SIv-nj73Ebo_aem_0qrAWsLYz7s-6nUdBdGrJw
PHISH 0.995 https://hgjhgkjhlkjhghg.weebly.com/
ok    0.052 https://morluto.gi

In [ ]:
# the step the earlier notebook was missing: liveness (+ VirusTotal, if VT_API_KEY is set) for URLs monitor flagged as phishing
!python freephish_old.py analyze --log detections.jsonl --obs observations.jsonl

analyze: checked liveness for 14 URL(s) flagged phishing; skipped 6 not flagged phishing and 0 older than --max-age-h (168h). observations.jsonl now has 28 line(s) total.


In [ ]:
import pandas as pd, json
det = pd.DataFrame([json.loads(l) for l in open("detections.jsonl")])
obs = pd.DataFrame([json.loads(l) for l in open("observations.jsonl")])
last = obs.sort_values("t").groupby("url").tail(1)[["url", "alive"]]
out = det.merge(last, on="url", how="left")
out = out[out["alive"].notna()]
out["alive"] = out["alive"].map({True: "alive", False: "dead"})
print(out[["url", "p_phish", "phishing", "alive"]].to_string(index=False))

                                                                                                                                                                                                                                                            url  p_phish  phishing alive
                                                                                                                                                                                                                   https://munivirtualenlinea.godaddysites.com/ 0.747116      True alive
                                                                                                                                                                                                                    https://munivirtualenlinea.godaddysites.com 0.829755      True alive
 https://lesusdelgado4.wixsite.com/mike-smith-web-servi?fbclid=IwY2xjawVbDiNleHRuA2FlbQIxMQBwZG9mAWJyaWQRMWJ3MWh1aXFYM2xkcTQwWGNzcnRjBmFwcF9pZBAyMjIwMzkxNzg4

## Stage 5: GAN attack on the enhanced model (and optionally the original)
No surrogate and no classifier inside the GAN itself: the discriminator only separates real feature vectors from the generator's output, the generator only tries to fool the discriminator. Only `url`/`html`/`freephish` features may change by default; `has_login_form`/`n_password_fields` are protected.

In [ ]:
!pip install -q torch
import os
need = ["freephish_old.py", "freephish_enhanced.py", "freephish_gan.py", "sample.parquet", "model.joblib", "model_enh.joblib",
        "results_baseline.predictions.csv", "results_enhanced.predictions.csv"]
print({f: os.path.exists(f) for f in need})

{'freephish_old.py': True, 'freephish_enhanced.py': True, 'freephish_gan.py': True, 'sample.parquet': True, 'model.joblib': True, 'model_enh.joblib': True, 'results_baseline.predictions.csv': True, 'results_enhanced.predictions.csv': True}


### 5a. `adv` — accuracy/precision/recall/F1 under attack (original vs enhanced)

In [ ]:
!python freephish_gan.py adv --csv sample.parquet \
    --original-model model.joblib --enhanced-model model_enh.joblib \
    --seeds 1 2 3 --epsilon 1.0 --epochs 150 \
    --cache-test feat_gan_test.npz --cache-train feat_adv_train.npz --out adv_results.json

test pages: 1392 (from your earlier run)
label values found: ['benign', 'phish']
positive (phishing) label(s): {'phish'}
loaded cached features: feat_gan_test.npz
label values found: ['benign', 'phish']
positive (phishing) label(s): {'phish'}
training pool: 4651 pages (2328 phishing / 2323 benign)
loaded cached features: feat_adv_train.npz
training attack GAN, seed 1 ...
training attack GAN, seed 2 ...
training attack GAN, seed 3 ...

RESULTS UNDER GAN ATTACK (mean +- sd over seeds [1, 2, 3]; test pages: 1392)
Model                                     Accuracy       Precision          Recall              F1
Original                             0.775 +-0.020   0.906 +-0.006   0.588 +-0.042   0.713 +-0.033
Enhanced                             0.797 +-0.025   0.937 +-0.005   0.614 +-0.052   0.741 +-0.040

saved adv_results.json (+ .csv)


### 5b. `retrain` — simple before/after on the clean test set
Trains the GAN once, adds the adversarial pages to the enhanced model's training data, retrains, and compares to the clean (non-adversarial) baseline it reads automatically from `results_enhanced.json`.

In [ ]:
!python freephish_gan.py retrain --csv sample.parquet --enhanced-model model_enh.joblib \
    --epsilon 1.0 --epochs 150 --cache-test feat_gan_test.npz --cache-train feat_adv_train.npz --out retrain_results.json --save-model model_adv.joblib

test pages: 1392 (from your earlier Stage 2 run)
label values found: ['benign', 'phish']
positive (phishing) label(s): {'phish'}
loaded cached features: feat_gan_test.npz
label values found: ['benign', 'phish']
positive (phishing) label(s): {'phish'}
training pool: 4651 pages (2328 phishing / 2323 benign)
loaded cached features: feat_adv_train.npz
training the GAN (no surrogate) on the training pool ...
generated 2328 adversarial phishing rows from the training pool

Enhanced (on the clean, unattacked test set):
  acc 0.944  prec 0.958  rec 0.923  f1 0.940

retraining the enhanced model on clean training pool + adversarial rows ...

Enhanced + adversarial training (on the SAME clean, unattacked test set):
  acc 0.945  prec 0.968  rec 0.914  f1 0.940
saved adversarially-trained model: model_adv.joblib

saved retrain_results.json


In [ ]:
from google.colab import files
for f in ["model.joblib", "model_enh.joblib", "model_adv.joblib"]:
    if os.path.exists(f):
        files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>